# Exercise_4

### 0. Resrobot time table API
  a) Make a function for doing get request for time tables API in resrobot as we didn't have a function in the lecture notes.

  b) Find the number of transports arriving to Göteborg centralstationen.

  c) Find the number of transports departuring from Göteborg centralstationen.

  d) Find the trams departuring from Göteborg centralstationen, their destinations and time. Note that trams in Swedish is "spårvagn" or in the dataset denoted as "Spårväg".

  e) See if you can plot in a map points corresponding to directions of each departuring tram.

  f) Do more EDA on this data to find things that you are interested in.

In [8]:
from dotenv import load_dotenv
import os
import requests

load_dotenv()

API_KEY = os.getenv("API_KEY1")

url1 = f"https://api.resrobot.se/v2.1/trip?format=json&originId=740000002&destId=740000003&passlist=true&showPassingPoints=true&accessId={API_KEY}"


url_stopId = f"https://api.resrobot.se/v2.1/departureBoard?id=740000002&format=json&accessId={API_KEY}"

url_stopName = f"https://api.resrobot.se/v2.1/location.name?input=Göteborg&format=json&accessId={API_KEY}"

url_arrivals = f"https://api.resrobot.se/v2.1/arrivalBoard?id=740000002&format=json&accessId={API_KEY}"

response = requests.get(url_stopId)
result = response.json()
result.keys()

dict_keys(['Departure', 'TechnicalMessages', 'serverVersion', 'dialectVersion', 'planRtTs', 'requestId'])

In [9]:
result["Departure"][2]

{'JourneyDetailRef': {'ref': '1|10430|1|1|14012025'},
 'JourneyStatus': 'P',
 'ProductAtStop': {'icon': {'res': 'prod_gen'},
  'operatorInfo': {'name': 'Västtrafik',
   'nameS': '279',
   'nameN': '279',
   'nameL': 'Västtrafik',
   'id': '279'},
  'name': 'Länstrafik - Buss 841',
  'internalName': 'Länstrafik - Buss 841',
  'displayNumber': '841',
  'num': '841',
  'line': '841',
  'lineId': '1279484100001',
  'catOut': 'BLT',
  'catIn': 'BLT',
  'catCode': '7',
  'cls': '128',
  'catOutS': 'BLT',
  'catOutL': 'Länstrafik - Buss',
  'operatorCode': '279',
  'operator': 'Västtrafik',
  'admin': '279___',
  'matchId': '841;4841;28'},
 'Product': [{'icon': {'res': 'prod_gen'},
   'operatorInfo': {'name': 'Västtrafik',
    'nameS': '279',
    'nameN': '279',
    'nameL': 'Västtrafik',
    'id': '279'},
   'name': 'Länstrafik - Buss 841',
   'internalName': 'Länstrafik - Buss 841',
   'displayNumber': '841',
   'num': '841',
   'line': '841',
   'lineId': '1279484100001',
   'catOut': 'BLT

In [10]:
def get_location_name(location):
    url_stopName = f"https://api.resrobot.se/v2.1/location.name?input={location}&format=json&accessId={API_KEY}"

    result_name = requests.get(url_stopName)
    return result_name.json().get("stopLocationOrCoordLocation")

location_name = get_location_name("Trätorget")
print(location_name)

[{'StopLocation': {'productAtStop': [{'icon': {'res': 'prod_gen'}, 'cls': '128'}], 'timezoneOffset': 60, 'id': 'A=1@O=Trätorget (Göteborg kn)@X=12053177@Y=57715439@U=1@L=740059660@B=1@p=1736829392@', 'extId': '740059660', 'name': 'Trätorget (Göteborg kn)', 'lon': 12.053177, 'lat': 57.715439, 'weight': 540, 'products': 128, 'minimumChangeDuration': 'PT0S'}}, {'CoordLocation': {'links': [{'link': [{'rel': 'refine', 'href': 'https://api.resrobot.se/v2.1/location.name?input=Tr%C3%A4torget%2C+Falk%C3%B6ping&refineId=A%3D2%40O%3DTr%C3%A4torget%2C+Falk%C3%B6ping%40X%3D13554412%40Y%3D58160352%40U%3D174%40b%3D980141995%40B%3D1%40p%3D1479298166%40&type=A'}]}], 'icon': {'res': 'loc_addr'}, 'id': 'A=2@O=Trätorget, Falköping@X=13554412@Y=58160352@U=174@b=980141995@B=1@p=1479298166@', 'name': 'Trätorget, Falköping', 'type': 'ADR', 'lon': 13.554412, 'lat': 58.160352, 'refinable': True}}, {'CoordLocation': {'links': [{'link': [{'rel': 'refine', 'href': 'https://api.resrobot.se/v2.1/location.name?input

In [11]:
location_name[0].get("StopLocation")

{'productAtStop': [{'icon': {'res': 'prod_gen'}, 'cls': '128'}],
 'timezoneOffset': 60,
 'id': 'A=1@O=Trätorget (Göteborg kn)@X=12053177@Y=57715439@U=1@L=740059660@B=1@p=1736829392@',
 'extId': '740059660',
 'name': 'Trätorget (Göteborg kn)',
 'lon': 12.053177,
 'lat': 57.715439,
 'weight': 540,
 'products': 128,
 'minimumChangeDuration': 'PT0S'}

In [12]:
station_id = location_name[0].get("StopLocation", {}).get("extId")
print(station_id)

740059660


# A

In [13]:
def get_departures(location_ids):
    url_departures = f"https://api.resrobot.se/v2.1/departureBoard?id={location_ids}&format=json&accessId={API_KEY}"
    response = requests.get(url_departures)
    data = response.json()
    departures = data.get("Departure", [])
    return departures

In [14]:
import re
station = get_location_name("Göteborg C")
def get_timetable():
    if station:
        station_name = station[0]["StopLocation"]["name"]
        station_id_raw = station[0]["StopLocation"]["id"]

        match = re.search(r"L=(\d+)", station_id_raw)
        if match:
            station_id = match.group(1)
        else:
            print("Kunde inte extrahera stationens ID.")
            station_id = None
        print(f"Stationens namn: {station_name}")
        print(f"Stations id: {station_id}")

        departures = get_departures(station_id)
        if departures:
            for departure in departures:
                print(f"Tid: {departure['time']}, Destination: {departure['direction']}")

            num_dep = len(departures)
            print(f"\nAntal transporter som avgår från {station_name}: {num_dep}")
        else:
            print("Inga avgångar")
    else:
        print("Stationen hittades inte")

get_timetable()

Stationens namn: Göteborg Centralstation
Stations id: 740000002
Tid: 15:23:00, Destination: Göteborg Eriksbergstorget
Tid: 15:23:00, Destination: Merkuriusgatan (Göteborg kn)
Tid: 15:23:00, Destination: Hinnebäcksgatan (Göteborg kn)
Tid: 15:23:00, Destination: Särö centrum (Kungsbacka kn)
Tid: 15:23:00, Destination: Kortedala Aprilgatan (Göteborg kn)
Tid: 15:23:00, Destination: Göteborg Varmfrontsgatan
Tid: 15:23:00, Destination: Göteborg Heden
Tid: 15:23:00, Destination: Sävedalen Ljungkullen (Partille kn)
Tid: 15:23:00, Destination: Torslandakrysset (Göteborg kn)
Tid: 15:24:00, Destination: Höga hallar (Härryda kn)
Tid: 15:24:00, Destination: Kungälv resecentrum
Tid: 15:24:00, Destination: Fyrktorget (Göteborg kn)
Tid: 15:24:00, Destination: Stenared (Göteborg kn)
Tid: 15:25:00, Destination: Hornkamsgatan (Göteborg kn)
Tid: 15:25:00, Destination: Göteborg Heden
Tid: 15:26:00, Destination: Höga hallar (Härryda kn)
Tid: 15:26:00, Destination: Kungälv resecentrum
Tid: 15:26:00, Destinat

In [15]:
station = get_location_name(input("Hållplats: "))
get_timetable()

Stationens namn: Trätorget (Göteborg kn)
Stations id: 740059660
Tid: 15:22:00, Destination: Hinnebäcksgatan (Göteborg kn)
Tid: 15:27:00, Destination: Göteborg Östra sjukhuset
Tid: 15:32:00, Destination: Göteborg Östra sjukhuset
Tid: 15:32:00, Destination: Hinnebäcksgatan (Göteborg kn)
Tid: 15:38:00, Destination: Göteborg Östra sjukhuset
Tid: 15:41:00, Destination: Göteborg Östra sjukhuset
Tid: 15:42:00, Destination: Hinnebäcksgatan (Göteborg kn)
Tid: 15:47:00, Destination: Göteborg Östra sjukhuset
Tid: 15:52:00, Destination: Göteborg Östra sjukhuset
Tid: 15:52:00, Destination: Hinnebäcksgatan (Göteborg kn)
Tid: 15:57:00, Destination: Göteborg Östra sjukhuset
Tid: 16:02:00, Destination: Göteborg Östra sjukhuset
Tid: 16:02:00, Destination: Hinnebäcksgatan (Göteborg kn)
Tid: 16:07:00, Destination: Göteborg Östra sjukhuset
Tid: 16:12:00, Destination: Göteborg Östra sjukhuset
Tid: 16:12:00, Destination: Hinnebäcksgatan (Göteborg kn)
Tid: 16:17:00, Destination: Göteborg Östra sjukhuset
Tid: 

# B


In [16]:
def get_arrivials(location_ids):
    url_arrivals = f"https://api.resrobot.se/v2.1/arrivalBoard?id={location_ids}&format=json&accessId={API_KEY}"
    response = requests.get(url_arrivals)

    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        print(response.text)
        return None
    
    data_arrivals = response.json()
    arrivals = data_arrivals.get("Arrival", [])
    return arrivals
get_arrivials("Trätorget")

Error: 400
{"serverVersion":"2.45.1","dialectVersion":"2.45","errorCode":"SVC_LOC","errorText":"location missing or invalid (LOCATION).","internalErrorCode":"LOCATION","internalErrorText":"HCI Service: location missing or invalid","internalErrorTextOut":"Det uppstod ett internt fel under sökningen","requestId":"default-request-id"}


In [17]:
station = get_location_name("Göteborg C")
def get_timetable_arrivals():
    if station:
        station_name = station[0]["StopLocation"]["name"]
        station_id_raw = station[0]["StopLocation"]["id"]

        match = re.search(r"L=(\d+)", station_id_raw)
        if match:
            station_id = match.group(1)
        else:
            print("Kunde inte extrahera stationens ID.")
            station_id = None
        print(f"Stationens namn: {station_name}")
        print(f"Stations id: {station_id}")

        arrivals = get_arrivials(station_id)
        if arrivals:
            print(f"Ankomster till {station_name}:")
            for arrival in arrivals:
                time = arrival.get('time', 'N/A')
                origin = arrival.get('origin', 'N/A')
                transport = arrival.get('ProductAtStop', {}).get('displayNumber', 'Okänt fordon')
                print(f"Tid: {time}, Avgång: {origin}, FordonsNr: {transport}")

            num_arrivals = len(arrivals)
            print(f"\nAntal transporter som anländer till {station_name}: {num_arrivals}")
        else:
            print("Inga avgångar")
    else:
        print("Stationen hittades inte")

get_timetable_arrivals()

Stationens namn: Göteborg Centralstation
Stations id: 740000002
Ankomster till Göteborg Centralstation:
Tid: 15:23:00, Avgång: Fyrktorget (Göteborg kn), FordonsNr: 16
Tid: 15:23:00, Avgång: Göteborg Eketrägatan, FordonsNr: 21
Tid: 15:23:00, Avgång: Göteborg Östra sjukhuset, FordonsNr: 17
Tid: 15:23:00, Avgång: Gråbo busstation (Lerum kn), FordonsNr: X3
Tid: 15:23:00, Avgång: Göteborg Varmfrontsgatan, FordonsNr: 6
Tid: 15:23:00, Avgång: Kortedala Aprilgatan (Göteborg kn), FordonsNr: 6
Tid: 15:23:00, Avgång: Göteborg Heden, FordonsNr: 173
Tid: 15:23:00, Avgång: Amhult Resecentrum (Göteborg kn), FordonsNr: SVART
Tid: 15:23:00, Avgång: Göteborg Östra sjukhuset, FordonsNr: SVART
Tid: 15:24:00, Avgång: Kungälv resecentrum, FordonsNr: X4
Tid: 15:24:00, Avgång: Mölnlycke station (Härryda kn), FordonsNr: X4
Tid: 15:24:00, Avgång: Göteborg Lindholmen, FordonsNr: 16
Tid: 15:24:00, Avgång: Stenungsund station, FordonsNr: SNU
Tid: 15:24:00, Avgång: Kullavik hamn (Kungsbacka kn), FordonsNr: X3
Tid: 

# A

In [18]:
station = get_location_name("Göteborg C")

get_timetable()

Stationens namn: Göteborg Centralstation
Stations id: 740000002
Tid: 15:23:00, Destination: Göteborg Eriksbergstorget
Tid: 15:23:00, Destination: Merkuriusgatan (Göteborg kn)
Tid: 15:23:00, Destination: Hinnebäcksgatan (Göteborg kn)
Tid: 15:23:00, Destination: Särö centrum (Kungsbacka kn)
Tid: 15:23:00, Destination: Kortedala Aprilgatan (Göteborg kn)
Tid: 15:23:00, Destination: Göteborg Varmfrontsgatan
Tid: 15:23:00, Destination: Göteborg Heden
Tid: 15:23:00, Destination: Sävedalen Ljungkullen (Partille kn)
Tid: 15:23:00, Destination: Torslandakrysset (Göteborg kn)
Tid: 15:24:00, Destination: Höga hallar (Härryda kn)
Tid: 15:24:00, Destination: Kungälv resecentrum
Tid: 15:24:00, Destination: Fyrktorget (Göteborg kn)
Tid: 15:24:00, Destination: Stenared (Göteborg kn)
Tid: 15:25:00, Destination: Hornkamsgatan (Göteborg kn)
Tid: 15:25:00, Destination: Göteborg Heden
Tid: 15:26:00, Destination: Höga hallar (Härryda kn)
Tid: 15:26:00, Destination: Kungälv resecentrum
Tid: 15:26:00, Destinat

# D Find the number of transports departuring from Göteborg centralstationen.

In [19]:
def get_departures_with_filter(station_name, transport_filter="Spårväg"):
    station = get_location_name(station_name)
    if station:
        station_name = station[0]["StopLocation"]["name"]
        station_id_raw = station[0]["StopLocation"]["id"]

        # Extrahera stationens ID
        match = re.search(r"L=(\d+)", station_id_raw)
        if match:
            station_id = match.group(1)
        else:
            print("Kunde inte extrahera stationens ID.")
            return

        print(f"Stationens namn: {station_name}")
        print(f"Stations id: {station_id}")

        # Hämta avgångar
        departures = get_departures(station_id)
        if departures:
            print(f"Avgångar från {station_name}:")
            for departure in departures:
                product_info = departure.get('ProductAtStop', {})
                category = product_info.get('name', 'N/A')

                # Filtrera om filter är satt
                if transport_filter is None or transport_filter in category:
                    time = departure.get('time', 'N/A')
                    direction = departure.get('direction', 'N/A')
                    transport = product_info.get('displayNumber', 'Okänt fordon')
                    typ = departure.get('catOutL')
                    print(f"Tid: {time}, Destination: {direction}, FordonsNr: {transport}")
        else:
            print("Inga avgångar hittades.")
    else:
        print("Stationen hittades inte")


In [20]:
get_departures_with_filter("Göteborg C", transport_filter="Spårväg")


Stationens namn: Göteborg Centralstation
Stations id: 740000002
Avgångar från Göteborg Centralstation:
Tid: 15:23:00, Destination: Kortedala Aprilgatan (Göteborg kn), FordonsNr: 6
Tid: 15:23:00, Destination: Göteborg Varmfrontsgatan, FordonsNr: 6
Tid: 15:33:00, Destination: Göteborg Varmfrontsgatan, FordonsNr: 6
Tid: 15:35:00, Destination: Kortedala Aprilgatan (Göteborg kn), FordonsNr: 6
Tid: 15:42:00, Destination: Göteborg Varmfrontsgatan, FordonsNr: 6
Tid: 15:44:00, Destination: Kortedala Aprilgatan (Göteborg kn), FordonsNr: 6
Tid: 15:52:00, Destination: Kortedala Aprilgatan (Göteborg kn), FordonsNr: 6
Tid: 15:53:00, Destination: Göteborg Varmfrontsgatan, FordonsNr: 6
Tid: 16:01:00, Destination: Kortedala Aprilgatan (Göteborg kn), FordonsNr: 6
Tid: 16:03:00, Destination: Göteborg Varmfrontsgatan, FordonsNr: 6
Tid: 16:09:00, Destination: Kortedala Aprilgatan (Göteborg kn), FordonsNr: 6
Tid: 16:14:00, Destination: Göteborg Varmfrontsgatan, FordonsNr: 6
Tid: 16:17:00, Destination: Korte

# E See if you can plot in a map points corresponding to directions of each departuring tram.

In [21]:
import sys
sys.path.append(r'C:\Users\Sandra\AppData\Local\Programs\Python\Python311\Lib\site-packages')

import folium

In [27]:
departures = get_departures(station_id)
print(departures[0])

{'JourneyDetailRef': {'ref': '1|57520|1|1|14012025'}, 'JourneyStatus': 'P', 'ProductAtStop': {'icon': {'res': 'prod_gen'}, 'operatorInfo': {'name': 'Västtrafik', 'nameS': '279', 'nameN': '279', 'nameL': 'Västtrafik', 'id': '279'}, 'name': 'Länstrafik - Buss 17', 'internalName': 'Länstrafik - Buss 17', 'displayNumber': '17', 'num': '17', 'line': '17', 'lineId': '1279501700001', 'catOut': 'BLT', 'catIn': 'BLT', 'catCode': '7', 'cls': '128', 'catOutS': 'BLT', 'catOutL': 'Länstrafik - Buss', 'operatorCode': '279', 'operator': 'Västtrafik', 'admin': '279___', 'matchId': '17;5017;149'}, 'Product': [{'icon': {'res': 'prod_gen'}, 'operatorInfo': {'name': 'Västtrafik', 'nameS': '279', 'nameN': '279', 'nameL': 'Västtrafik', 'id': '279'}, 'name': 'Länstrafik - Buss 17', 'internalName': 'Länstrafik - Buss 17', 'displayNumber': '17', 'num': '17', 'line': '17', 'lineId': '1279501700001', 'catOut': 'BLT', 'catIn': 'BLT', 'catCode': '7', 'cls': '128', 'catOutS': 'BLT', 'catOutL': 'Länstrafik - Buss', 

In [49]:
import webbrowser

def plot_tram_arrival_destination(station_name):
    station = get_location_name(station_name)

    if station:
        station_name = station[0]["StopLocation"]["name"]
        station_id_raw = station[0]["StopLocation"]["id"]

        match = re.search(r"L=(\d+)", station_id_raw)
        if match:
            station_id = match.group(1)
        else:
            print("Kan inte hämta stations ID")
            return
        
        print(f"Stationens namn: {station_name}")
        print(f"Stationens namn: {station_id}")


        departures = get_departures(station_id)
        
        if departures:
            station_lat = station[0]["StopLocation"]["lat"] / 1e6
            station_lon = station[0]["StopLocation"]["lon"] / 1e6

            tram_map = folium.Map(location=[station_lat, station_lon], zoom_start=14)

            print(f"Plotting tram directions from {station_name}...")

            for departure in departures:
                product_info = departure.get("ProductAtStop", {})
                category = product_info.get("name", "N/A")

                #print(category)

                if "Spårväg" in category:
                    direction = departure.get("direction", "N/A")
                    time = departure.get("time", "N/A")

                    destination_lat = departure.get("lat")
                    destination_lon = departure.get("lon")

                    #print(destination_lon)

                    if destination_lat is None or destination_lon is None:
                        print(f"Missing coordinates for departure: {departure}")
                        continue

                    folium.Marker(location = [destination_lat, destination_lon],
                                popup = f"Destination: {direction}<br>Tid: {time}", icon=folium.Icon(color="blue", icon="info_sign")).add_to(tram_map)
                    
            map_file = "tram_departure_map.html"
            tram_map.save(map_file)
            print(f"Map saved as {map_file}")
            webbrowser.open(map_file)
        else:
            print("No tram departures found.")
    else:
        print("No station found...")

plot_tram_arrival_destination("Göteborg C")


Stationens namn: Göteborg Centralstation
Stationens namn: 740000002
Plotting tram directions from Göteborg Centralstation...
Map saved as tram_departure_map.html
